### Importing libraries

In [86]:
# # %pip install ddgs trafilatura -q
# %pip install openai-agents 

In [87]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
from openai import OpenAI
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool
from agents import trace
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
import base64
import os
import time


In [88]:
REASONING_MODEL = "o4-mini"
MODEL = "gpt-4.1-mini"

### Setup

In [89]:
# api
load_dotenv()

OpenAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key= OpenAI_API_KEY)

if OpenAI_API_KEY is None:
    raise Exception ("No API Key for Open AI")
else:
    print(OpenAI_API_KEY[:8])

sk-proj-


#### Step 1: Define Tools

In [90]:
#search web tool

@function_tool
def search_web(query:str):
    """Seach the web using ddgs browser. Return the top 3 results"""
    #initialize a ddgs object:
    ddgs = DDGS()
    results = ddgs.text(query, max_results = 3)
    print(f'\u2705 Used search_web to get results from this query: {query}')
    #returning in json bc LLM doesnt work with dictionaries, so json converts to text
    return json.dumps(results, indent= 2)

In [91]:
#Fetching the information from the search results. Scraping the url
@function_tool
def fetch_url(url:str):
    """Extracting text from fetched from web result url"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f'\u2705 Fetch URL: got text {len(text)} characters and this is the url {url[:60]}')
            return text
        else:
            print(f'\u274C Failed to fetch or extract text.')
    return f'Could not extract text from {url}. Try a different source.'

### Step 2: Agents

#### The Agents tells the LLM who it is and how to behave. The key things:
* - What its job is
* - What tools it has
* - What process to follow
* - What output format to produe

### Research Agent

In [92]:

RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.


You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page


***IMPORTANT:
After each search using search_web, you MUST first explain your reasoning:
- Which URLs look most relevant and why
- Which ones you will fetch and why
- Which ones you are skipping and why
Only AFTER writing out your reasoning should you call fetch_url.
- MAKE SURE IF YOU CALL SEARCH_WEB multiple times why you decided to do multiple search_webs.
***

IMPORTANT: DO NOT just tool call over and over again. You need to THINK before doing a tool call. This is not a comprehensive brief if you do not think before tool calls. I want to see you working through deciding what are the
best sources or urls to use. You NEED AT LEAST 3 sources. Make sure to think through which is the best 3 of all your sources. Try and think between tool calls to determine if you need to search the web more for a better topic search
or fetch a different url. You should use your discernment to try and think through the research brief to create a concise 2 paragraph brief, with cited sources. 


Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

#This is used instead of the def run_research_agent loop we usually make:
research_agent = Agent(
    name = "Research Agent",
    instructions= RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)


### Image Generator Agent

In [93]:
#initializing open ai client 
openai_client = OpenAI()

@function_tool
def generate_image(prompt: str) -> str:
    """Generate an image using gpt-image-1."""

    print(f"🎨 Generating image from: {prompt[:60]}")

    try:
        print("Calling gpt-image-1 with:")
        print("model = gpt-image-1")
        print("size = 1536x1024")
        print("quality = high")

        response = openai_client.images.generate(
            model="gpt-image-1",
            prompt=prompt,
            size="1536x1024",
            quality="high",
            n=1
        )

        image_bytes = base64.b64decode(response.data[0].b64_json)

        os.makedirs('generated_images', exist_ok = True)

        
        filepath = f"generated_images/{int(time.time())}.png"
        with open(filepath, "wb") as f:
            f.write(image_bytes)

        print("✅ Image generated successfully")

        return filepath

    except Exception as e:
        print("❌ IMAGE GENERATION ERROR")
        print(type(e).__name__)
        print(str(e))
        raise




In [94]:
#Image generator Agent prompt and initialization:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed image generation prompt for a hero image.

Rules for your image prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words
- Create an image that drives the message of topic home to the readers

Call generate_image exactly ONCE with your prompt. One image only.

IMPORTANT: generate_image returns a local file path (e.g. generated_images/1234567890.png).
When reporting the result, copy that file path EXACTLY character by character.
Do not modify, shorten, or paraphrase it. Do not refer to it as a URL.
"""

image_agent = Agent(
    name = "Image_Agent",
    model = MODEL,
    instructions = IMAGE_AGENT_PROMPT,
    tools = [generate_image]
)
image_agent_as_tool= image_agent.as_tool(tool_name = 'image_agent', tool_description = 'Generates a hero image from a research topic and content research brief. Pass the topic and content summary so far.', max_turns = 20)


### Step 3: Orchestrator Agent

In [95]:
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

IMPORTANT: Do not include your instructions to the agents, you must only include the reseach brief and image, that is all.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.

Once you have selected the best brief, you MUST use the image_agent tool to generate an image.
Use the research brief to supply the image agent with the topic and content it needs to generate the image.

Then decide which writer is best for this topic and research:
Journalist: investigative, thought-provoking writing, challenges assumptions and ask hard questions throughout. This writer is best with topics where a clear stance is needed, investigating or trying to figure out the truth about an event, or a creative written think piece.
Educator: assumes the reader is a novice, beginner, or is trying to learn a difficult educational topic. This writer is best with topics trying to learn about a subject matter or explanation of a hard concept.

Then handoff to chosen writer.

IMPORTANT: Do not write the article yourself you must pass it off to a writer.

"""
#IMPORTANT: When passing the image file path, copy it EXACTLY character by character.
# Do not modify, shorten, paraphrase, or convert it into a URL — it is a local file path, not a URL. -- not needed provided in convo history for orchestrator_agent

orchestrator_agent = Agent(
    name = "Orchestrator Agent",
    model = REASONING_MODEL,
    instructions = ORCHESTRATOR_AGENT_PROMPT,
    tools = [research_agent.as_tool(tool_name="research_agent", tool_description= "Researchs a topic and provides a comprehensive research brief.",  max_turns = 20), image_agent_as_tool ]

)

### Journalist Agent

In [96]:
Journalist_Prompt = RECOMMENDED_PROMPT_PREFIX + """

You are a investigative journalist who write thought-provoking, impactful research briefs and content summaries. 
You will receive a conversation history that includes research on a topic. 
You have a creative writing style that grips readers to keep reading more. 
Given a topic you will write a piece of content that is news-worthy but coherent grammatically and literature wise.

Do not overuse headers, only use 2 to 3 headers for one topic.

Your writing style contains these qualities:
- Punch lines that either convey humor, impactful to the reader, or thought-provoking
- Controversial takes are ok, as long as you are not writing something polatrizing views
- Ground all your writing to real facts and stories you have scraped from the web
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 


IMAGE HANDLING:
You will receive a local image file path (e.g. generated_images/1234567890.png) as part of your input.
This is a local file path, NOT a URL — do not modify, shorten, or convert it.
Embed it once near the top of the article as a markdown image, using the exact path:
![Hero image](generated_images/1234567890.png)
Use the actual path you were given, copied exactly, in place of the example above.

Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. 
"""

Journalist_Agent = Agent(
    name = 'Journalist_Agent',
    instructions = Journalist_Prompt,
    model = REASONING_MODEL
)



### Educator Agent

In [97]:
#Eductor Prompt:

Educator_Prompt =  RECOMMENDED_PROMPT_PREFIX + """You are an Educator Agent. You will assume the reader is novice in the subject mattter and will write a research brief with writing style where you take a complex topic
and make it easy to understand. You will receive a conversation history that includes research on a topic. 

Do not overuse headers, only use 2 to 3 headers for one topic.

Writing Style:
-   Assumes the reader is a complete beginner
- Quote sources and reference specific data points 
-   Begin your writing with the how the topic came to be or the foundation learning components of the research topic and 
then conclude with where the topic is today or the complexity of the topic in present day.
- If the topic is trying to learn about a subject matter or explanation of a hard concept, explain to the readers the topic 'like they are 5 years old' or that they are laymen who has never 
heard of such complex concepts.
- Make sure to have small math examples if a math topic, small dataset that the reader can follow if the numbers are complex. Use examples to help reader really understand subject that they could write their own 
research paper.


IMAGE HANDLING:
You will receive a local image file path (e.g. generated_images/1234567890.png) as part of your input.
This is a local file path, NOT a URL — do not modify, shorten, or convert it.
Embed it once near the top of the article as a markdown image, using the exact path:
![Hero image](generated_images/1234567890.png)
Use the actual path you were given, copied exactly, in place of the example above.

Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. 

"""

#Initializing the agent 
Educator_Agent = Agent(
    name = 'Educator_Agent',
    instructions = Educator_Prompt,
    model = REASONING_MODEL
)

#### Local Testing just to see how the Journalist Agent works

In [98]:
#This just for our local testing to see how the journalist agent works

#  with trace("Testing Journalist_Agent", group_id= 'Testing Journalist Agent'):
#     result = await Runner.run(
#         Journalist_Agent,
#         input = "Research the following topic and produce a comprehensive research brief: Will AI jobs disappear soon",
#         max_turns = 30
#     )

# with trace('Testing Educator Agent', group_id= 'Testing Educator Agent'):
#     result = await Runner.run(
#         Educator_Agent,
#         input = 'The research brief topic is: understanding arrays versus lists in python',
#         max_turns = 20
#     )

In [99]:
#This just for our local testing to see how the journalist agent works
# print(f'Agent: {result.last_agent.name}')
# print(f'-----')
# display(Markdown(result.final_output))

# #testing educator:
# print(f' Agent: {result.last_agent.name}')
# display(Markdown(result.final_output))

#### Testing Understanding: agent used as tool v. search_web as tool (same)

In [100]:
# test_tool = research_agent.as_tool(tool_name="Research Agent", tool_description= "Researchs a topic and provides a comprehensive research brief.")
# pprint(test_tool)


In [101]:
# pprint(search_web)

### Step 4: Updating the Orchestrator Agent

In [102]:
#I can also change the handoffs as an argument in the orchestrator agent = AGENT(handoffs =)
from agents import handoff

orchestrator_agent.handoffs = [handoff(agent= Journalist_Agent), handoff(agent= Educator_Agent)]

In [1]:
ORCHESTRATOR_AGENT_PROMPT = RECOMMENDED_PROMPT_PREFIX + """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate agents to produce a high-quality article. 
IGNORE the tools available to you.
Your agents are specialists and should be doing the work, you are the manager.

As soon as you have the topic, RIGHT AWAY decide which writer is best for the topic.

Then decide which writer is best for this topic and research:
Journalist: investigative, thought-provoking writing, challenges assumptions and ask hard questions throughout. This writer is best with topics where a clear stance is needed, investigating or trying to figure out the truth about an event, or a creative written think piece.
Educator: assumes the reader is a novie, beginner, or is trying to learn a difficult educational topic. This writer is best with topics trying to learn about a subject matter or explanation of a hard concept.

Then handoff to chosen writer.

IMPORTANT: Do not write the article yourself you must pass it off to a writer.

"""

NameError: name 'RECOMMENDED_PROMPT_PREFIX' is not defined

### Let's Run it!

In [105]:
with trace('Journalist Writer and Educator Writer w/ Handoff', group_id= 'Learning AI Engineering'):
    result = await Runner.run(
        orchestrator_agent,
        # input = "Research the following topic and produce a comprehensive research brief: how does one learn how to play pickleball at age 25?",
        input = "Write an article about: How to bake an apple pie from scratch",
        max_turns= 30
    )

✅ Used search_web to get results from this query: how to bake an apple pie from scratch ingredients and instructions
✅ Fetch URL: got text 9098 characters and this is the url https://littlespoonfarm.com/apple-pie-recipe/
✅ Fetch URL: got text 3878 characters and this is the url https://www.allrecipes.com/recipe/12324/apple-pie-i/
✅ Fetch URL: got text 9157 characters and this is the url https://www.attainable-sustainable.net/apple-pie-recipe/
✅ Used search_web to get results from this query: how to bake apple pie from scratch ingredient selection pastry techniques baking times variations serving suggestions
✅ Fetch URL: got text 9460 characters and this is the url https://www.simplyrecipes.com/recipes/old_fashioned_apple_pi
✅ Used search_web to get results from this query: apple pie variations and serving suggestions
✅ Fetch URL: got text 7888 characters and this is the url https://www.bettycrocker.com/recipes/apple-pie-recipe/c9a4ac
✅ Fetch URL: got text 15027 characters and this is t

In [51]:
# #research agent defined above and runner runs the runner


# with trace('Article Writer', group_id = 'Learning AI Engineering', metadata = {'model': MODEL, 'experiment_id' : '001' }):
#     result = await Runner.run(
#         orchestrator_agent,
#         input = "Research the following topic and produce a comprehensive research brief: How AI is will be used in Helathcare in 2040",
#         max_turns= 30
#     )


    

* Orchestrator Turn 1
→ call research_agent(input A)

* Research Agent runs internally
→ returns Brief A

* Orchestrator Turn 2
→ now sees Brief A
→ calls research_agent(input B)

* Research Agent runs internally
→ returns Brief B

* Orchestrator Turn 3
→ now sees Brief A + Brief B
→ compares them
→ outputs whichever one it thinks is better

In [106]:
print(f'Agent: {result.last_agent.name}')
print(f'-----')
display(Markdown(result.final_output))

Agent: Educator_Agent
-----


Here’s the selected research brief for “How to Bake an Apple Pie from Scratch,” followed by the hero image:

**Comprehensive Research Brief:**

1. Ingredient Selection  
- Apples: Mix tart and sweet varieties (Granny Smith, Jonagold, Fuji, Golden Delicious, Braeburn, Cortland, Honeycrisp). Avoid Red Delicious.  
- Spices: Cinnamon, nutmeg, allspice; optional cardamom or cloves.  
- Crust: All-purpose flour, cold butter or shortening, salt, cold water or sour cream.  
- Flavor Boosters: Vanilla extract, lemon juice or zest, brandy (optional).  

2. Pastry Techniques  
- Keep fat cold and handle dough minimally for flakiness.  
- Chill dough before rolling; roll bottom and top crust to ~1/8″ thickness.  
- Fit into a 9″ pie plate, seal edges, and vent top crust or lattice.  
- Brush with egg wash for gloss; use pie shield or foil to prevent edge burning.  

3. Baking Times & Temperatures  
- Standard 9″ pie: 375°F for 20 min, then 350°F for 60–90 min until crust is golden and filling bubbles.  
- Deep dish: 400°F for 20 min, then 375°F for 45–55 min.  
- Aim for ~200°F internal filling.  
- Cool 1 hour (3 hours deep dish) to let filling set.  

4. Variations  
- Dutch-style with streusel topping.  
- Salted caramel apple pie with caramel drizzle.  
- Individual apple hand pies.  
- Add nuts (pecans, walnuts) or swap sweeteners (honey, maple syrup).  
- Gluten-free or alternative-spice versions.  

5. Serving Suggestions  
- Warm or room temperature.  
- Top with vanilla or cinnamon ice cream, whipped cream, or caramel sauce.  
- Pair with sharp cheddar cheese, coffee, or spiced cider.  

6. Storage & Freezing  
- Room temperature up to 2 days; fridge up to 7 days.  
- Freeze unbaked or baked pie tightly wrapped; bake frozen unbaked pie with extra time.  
- Reheat at 350°F for best texture.  

Sources:  
- SimplyRecipes (Old Fashioned Apple Pie)  
- Betty Crocker (Apple Pie Recipe)  
- RecipeToCook (UK Apple Pie)  
- IamWinfred.com (Deep Dish Apple Pie)

![Hero image](generated_images/1786934739.png)